# CAMUS — DRL Boundary Refinement

**Pick agent + structure in Cell 1.** Run cells **0 → 1 → 2** always. Then either §3 (dry-run, ~3 min) **or** §4 (full run, ~3 hr) — not both. Then §5 unpacks, §6 visualises.

## Kaggle setup
1. Attach: **CAMUS dataset** + **iteris-pkg** + **camus-baseline-outputs** (U-Net `camus_best.pt`)
2. Accelerator: **GPU T4 x2**, Persistence: **Files only**, Internet: **On**
3. For full run: Save Version → Save & Run All (Commit)

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'), '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Package at: {PKG_ROOT}')

## 1 · Configure — set agent + structure here

In [ ]:
# ══════════════════════════════════════════════════════════════════════
CFG_NAME     = 'camus_dqn.yaml'   # camus_dqn.yaml | camus_dueling.yaml | camus_ddpg.yaml
TARGET_CLASS = 1                  # 1=LV_endo, 2=LV_epi, 3=LA
# ══════════════════════════════════════════════════════════════════════

import yaml
from iteris.config import load_config
from iteris.utils  import get_device

with open(PKG_ROOT / 'configs' / CFG_NAME) as f:
    cfg = yaml.safe_load(f)
baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

cfg['target_class'] = TARGET_CLASS
cfg['checkpoint_dir'] = '/kaggle/working'

# Auto-detect CAMUS root and U-Net checkpoint
camus_roots = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_roots:
    baseline_cfg['data_root'] = str(camus_roots[0])
ckpt_cands = [p for p in Path('/kaggle/input').rglob('camus_best.pt')]
if ckpt_cands:
    cfg['baseline_checkpoint'] = str(ckpt_cands[0])

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({baseline_cfg["class_names"][cfg["target_class"]]})')
print(f'CAMUS data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Device          : {get_device()}')

## 2 · Warm-start — pre-compute U-Net init masks
~5 min for CAMUS. Caches U-Net predictions so DRL training doesn't repeat them.

In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.01),
)
print(f'\nSamples — train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')

init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
              for s in val_samples]
print(f'U-Net init Dice on val: mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')

## 3 · OPTIONAL — Dry-run sanity check (~3 min)
**Self-contained**: uses a `deepcopy(cfg)` and inline-sliced sample lists,
so it does NOT mutate `cfg`, `train_samples`, `val_samples`, or `test_samples`.
You can safely run this cell AND §4 below — §4 will still use the full data.

In [ ]:
# === DRY RUN — self-contained, leaves cfg + sample lists unchanged ===
import copy

_dry_cfg = copy.deepcopy(cfg)
_dry_cfg['train_steps']         = 600
_dry_cfg['prefill_steps']       = 100
_dry_cfg['buffer_size']         = 300
_dry_cfg['eval_every']          = 200
_dry_cfg['epsilon_decay_steps'] = 400
_dry_cfg['batch_size']          = 32

from iteris.drl_training import run_drl_training
_dry_result = run_drl_training(_dry_cfg, train_samples[:30], val_samples[:10])
print(f'\n✓ Dry run passed. Best val Dice (meaningless): {_dry_result["best_dice"]:.4f}')
print(f'  cfg["train_steps"] still = {cfg["train_steps"]}  (unchanged)')
print(f'  len(train_samples)       = {len(train_samples)}  (unchanged)')
print('  → Safe to run §4 below for the real training run.')

## 4 · Full training (~3 hr DDQN/Dueling, ~5 hr DDPG)
**Only run this OR §3, not both.** For the real commit, skip §3 entirely.

In [ ]:
from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']
best_dice = result['best_dice']

print(f'\n✓ Training complete. Best val final-Dice: {best_dice:.4f}')
print(f'  Checkpoint: {result["checkpoint"]}')

## 5 · Visualisation helper
Defines `replay_episode()` and `ENV_KW`. Used by all viz cells below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from iteris.env import SegmentationEnv, dice_score, hd95_px

ENV_KW = dict(
    action_type       = agent.action_type,
    max_steps         = cfg.get('max_steps', 20),
    shift_px          = cfg.get('shift_px', 2),
    sdt_clip          = cfg.get('sdt_clip', 20.0),
    reward_clip       = cfg.get('reward_clip', 1.0),
    cont_action_scale = cfg.get('cont_action_scale', 0.04),
)
CLASS_COLOR    = baseline_cfg['class_colors'][cfg['target_class']]
CMAP_SINGLE    = ListedColormap([CLASS_COLOR])
DISCRETE_NAMES = ['dilate', 'erode', '↑', '↓', '←', '→', 'no-op']

def replay_episode(sample):
    """Greedy rollout. Returns (mask_list, dice_list, hd95_list, action_list)."""
    env = SegmentationEnv(sample['image'], sample['gt_mask'], sample['init_mask'], **ENV_KW)
    state = env.reset()
    masks, actions = [env.mask.copy()], []
    while True:
        if agent.action_type == 'discrete':
            a = agent.select_action(state, epsilon=0.0)
        else:
            a = agent.select_action(state, explore=False)
        actions.append(a)
        state, _, done, _ = env.step(a)
        masks.append(env.mask.copy())
        if done:
            break
    return masks, list(env.dice_history), list(env.hd95_history), actions

# Pre-replay a small fixed subset — used by all viz cells (no re-computation)
N_VIZ = min(8, len(test_samples))
np.random.seed(0)
viz_idx = np.random.choice(len(test_samples), size=N_VIZ, replace=False).tolist()

print(f'Replaying {N_VIZ} test samples for visualisation...')
replays = []
for i in viz_idx:
    masks, dices, hd95s, actions = replay_episode(test_samples[i])
    replays.append(dict(
        sample=test_samples[i],
        masks=masks, dices=dices, hd95s=hd95s, actions=actions,
        init_d=dices[0], final_d=dices[-1], gain=dices[-1]-dices[0],
    ))

replays.sort(key=lambda r: r['gain'])
print(f'Done. ΔDice across {N_VIZ}: min {replays[0]["gain"]:+.4f}  '
      f'median {replays[N_VIZ//2]["gain"]:+.4f}  max {replays[-1]["gain"]:+.4f}')

## 6 · Sample comparison — best / median / worst
Reuses the cached replays — ~5 seconds.

In [ ]:
picks = [
    ('BEST gain',   replays[-1]),
    ('MEDIAN gain', replays[N_VIZ // 2]),
    ('WORST gain',  replays[0]),
]

fig, axes = plt.subplots(len(picks), 4, figsize=(16, 4*len(picks)))
for row, (label, r) in enumerate(picks):
    s, final_mask = r['sample'], r['masks'][-1]
    img, init, gt = s['image'], s['init_mask'], s['gt_mask']
    init_h  = hd95_px(init,       gt)
    final_h = hd95_px(final_mask, gt)

    cells = [
        ('Input',                       None,       None,        None),
        ('U-Net init',                  init,       r['init_d'], init_h),
        (f'{cfg["agent_type"]} refined', final_mask, r['final_d'], final_h),
        ('Ground Truth',                gt,         1.0,         0.0),
    ]
    for col, (title, mask, d, h) in enumerate(cells):
        ax = axes[row, col]
        ax.imshow(img, cmap='gray')
        if mask is not None:
            ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=CMAP_SINGLE, alpha=0.5)
        if d is not None:
            ax.set_title(f'{title}\nDice {d:.3f}  HD95 {h:.1f}px', fontsize=10)
        else:
            ax.set_title(f'{title}\n[{label}]  ΔDice {r["gain"]:+.4f}', fontsize=10)
        ax.axis('off')

cls_name = baseline_cfg['class_names'][cfg['target_class']]
plt.suptitle(f'CAMUS {cfg["agent_type"]} — {cls_name} (class {cfg["target_class"]})', fontsize=13)
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_comparison.png', dpi=150)
plt.show()

## 7 · Iteration playback — one sample, all 20 steps
The differentiator visualisation (feeds the UI's Iteration Playback page). Green contour = GT boundary.

In [ ]:
from scipy import ndimage

# Use the best-gain replay for the playback (most informative trajectory)
r = replays[-1]
s = r['sample']
gt_edge = s['gt_mask'] ^ ndimage.binary_erosion(s['gt_mask'], iterations=1)

n_steps = len(r['masks'])
ncols   = 5
nrows   = int(np.ceil(n_steps / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
axes = np.atleast_2d(axes).flatten()

for i, (mask, dice, hd) in enumerate(zip(r['masks'], r['dices'], r['hd95s'])):
    ax = axes[i]
    ax.imshow(s['image'], cmap='gray')
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=CMAP_SINGLE, alpha=0.55)
    ax.imshow(np.ma.masked_where(~gt_edge.astype(bool), gt_edge), cmap='Greens', alpha=0.8)

    if i == 0:
        title = f't=0 (init)\nDice {dice:.3f}'
    else:
        a = r['actions'][i-1]
        if agent.action_type == 'discrete':
            a_str = DISCRETE_NAMES[int(a)]
        else:
            a_str = f'({a[0]*256:+.1f},{a[1]*256:+.1f})px'
        title = f't={i}  {a_str}\nDice {dice:.3f}  Δ {dice-r["dices"][i-1]:+.4f}'
    ax.set_title(title, fontsize=8)
    ax.axis('off')

for j in range(n_steps, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'{cfg["agent_type"]} iteration playback — '
             f'{s.get("patient","?")} {s.get("view","")} {s.get("phase","")}  '
             f'(green = GT boundary)', fontsize=12)
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_playback.png', dpi=150)
plt.show()

## 8 · Behaviour analysis — trajectories + action distribution
Reveals whether the agent makes monotonic progress vs. oscillates, and which actions it favours.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1 — Dice trajectories
ax = axes[0]
for r in replays:
    ax.plot(r['dices'], alpha=0.5, lw=1)
max_len = max(len(r['dices']) for r in replays)
padded = np.stack([np.pad(r['dices'], (0, max_len-len(r['dices'])),
                          constant_values=r['dices'][-1]) for r in replays])
ax.plot(padded.mean(axis=0), color='black', lw=2.5, label='mean')
ax.set(xlabel='Step', ylabel='Dice', title='Per-sample Dice trajectory')
ax.legend(); ax.grid(alpha=0.3)

# Panel 2 — HD95 trajectories
ax = axes[1]
for r in replays:
    valid = [v for v in r['hd95s'] if not np.isnan(v)]
    ax.plot(valid, alpha=0.5, lw=1)
ax.set(xlabel='Step', ylabel='HD95 (px)', title='Per-sample HD95 trajectory')
ax.grid(alpha=0.3)

# Panel 3 — Action distribution
ax = axes[2]
all_actions = [a for r in replays for a in r['actions']]
if agent.action_type == 'discrete':
    counts = np.bincount(all_actions, minlength=7)
    ax.bar(range(7), counts / counts.sum())
    ax.set_xticks(range(7))
    ax.set_xticklabels(DISCRETE_NAMES, rotation=30)
    ax.set(ylabel='Frequency', title='Action distribution')
    ax.grid(alpha=0.3, axis='y')
else:
    arr = np.array(all_actions)
    ax.scatter(arr[:, 1]*256, arr[:, 0]*256, alpha=0.5, s=20)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    sc = cfg.get('cont_action_scale', 0.04) * 256
    ax.set_xlim(-sc*1.1, sc*1.1); ax.set_ylim(-sc*1.1, sc*1.1)
    ax.set(xlabel='dx (px)', ylabel='dy (px)', title='Continuous action distribution')
    ax.grid(alpha=0.3); ax.set_aspect('equal')

plt.suptitle(f'{cfg["agent_type"]} behaviour analysis (class {cfg["target_class"]})')
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_behaviour.png', dpi=150)
plt.show()

print(f'\n── {cfg["agent_type"]} c{cfg["target_class"]} behaviour ──')
print(f'  Init  Dice avg : {np.mean([r["init_d"]  for r in replays]):.4f}')
print(f'  Final Dice avg : {np.mean([r["final_d"] for r in replays]):.4f}')
print(f'  Δ Dice avg     : {np.mean([r["gain"]   for r in replays]):+.4f}')
print(f'  Avg ep length  : {np.mean([len(r["dices"])-1 for r in replays]):.1f} steps')
if agent.action_type == 'discrete':
    print(f'  Most-used      : {DISCRETE_NAMES[np.argmax(counts)]} '
          f'({counts.max()/counts.sum()*100:.0f}%)')

## 9 · Learning curves (training history)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', linestyle='--', alpha=0.6)
ax1.plot(history['step'], history['final_dice_mean'], label=f'Final Dice ({cfg["agent_type"]})')
ax1.set(xlabel='Training step', ylabel='Mean Val Dice', title='Refinement learning curve')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['step'], history['delta_dice_mean'], color='C2')
ax2.axhline(0, color='k', lw=0.5)
ax2.set(xlabel='Training step', ylabel='Δ Dice (final − init)',
        title=f'Refinement gain ({cfg["agent_type"]} c{cfg["target_class"]})')
ax2.grid(alpha=0.3)

plt.suptitle(f'CAMUS {cfg["agent_type"]} — class {cfg["target_class"]} '
             f'({baseline_cfg["class_names"][cfg["target_class"]]})')
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_curves.png', dpi=150)
plt.show()

## 10 · Test-set evaluation + summary JSON
Runs greedy episodes on the full test set. Slow on the full 200 samples (~7 min) — only run for committed final runs, skip for dry runs.

In [ ]:
from iteris.drl_training import evaluate_agent
import json

test_metrics = evaluate_agent(agent, test_samples, env_kwargs=ENV_KW)
print(json.dumps(test_metrics, indent=2))

summary = {**test_metrics,
           'agent_type'   : cfg['agent_type'],
           'target_class' : cfg['target_class'],
           'class_name'   : baseline_cfg['class_names'][cfg['target_class']],
           'best_val_dice': float(best_dice)}
out = f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_test_results.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved → {out}')